### ============================================================================
### MULTIMODAL LATE FUSION (FEATURE-LEVEL) FOR BRAIN TUMOR DETECTION
### ============================================================================
### Author: Jeisson Parra Prieto 
### Description:
###   - Loads pre-trained MRI (two-stage) and CT (correlation) models.
###   - Extracts features from the penultimate layer of each model.
###   - Concatenates features and trains a fusion classifier on paired data.
###   - Evaluates performance against unimodal baselines.
###   - Generates Grad-CAM heatmaps for interpretability.
### ============================================================================


In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import re
from math import pi
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
import cv2
import matplotlib.cm as cm

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    # Paths to pre-trained models
    'MRI_STAGE1_PATH': './Saved_models/MRI_stage1_binary_model.keras',
    'MRI_STAGE2_PATH': './Saved_models/MRI_stage2_multiclass_model.keras',
    'CT_MODEL_PATH': './Saved_models/ct_correlation_model.keras',

    # Data directories (must contain train/val/test subfolders with MRI and CT)
    'MRI_DATA_ROOT': '/Users/jeissonparra/Documents/Master_s Degree Florida International University/Data Science & AI/Spring - 2026/Capstone/Datasets/Balanced_Multimodal/MRI',   
    'CT_DATA_ROOT': '/Users/jeissonparra/Documents/Master_s Degree Florida International University/Data Science & AI/Spring - 2026/Capstone/Datasets/Balanced_Multimodal/CT',    

    # Fusion model parameters
    'FUSION_INPUT_SHAPE_MRI': (128, 128, 4),
    'FUSION_INPUT_SHAPE_CT': (224, 224, 4),
    'NUM_CLASSES': 2,                # binary: normal vs tumor
    'BATCH_SIZE': 16,                 # smaller due to two inputs
    'EPOCHS': 50,
    'LEARNING_RATE': 1e-4,

    # Output paths
    'FUSION_MODEL_SAVE_PATH': './Saved_models/fusion_model.keras',
    'FIGURE_PATH': './Results/Fusion_figures',
    'RESULTS_PATH': './Results',
}

# Class names
CLASS_NAMES = ['Normal', 'Tumor']
STAGE2_CLASS_NAMES = ['Meningioma', 'Glioma', 'Pituitary']

# Colours for plots (consistent with your previous style)
COLORS = {
    'mri': '#3498DB',
    'ct': '#E67E22',
    'multimodal': '#9B59B6',
    'normal': '#2ECC71',
    'tumor': '#E74C3C',
    'background': '#F8F9FA',
    'meningioma': '#3498DB',
    'glioma': '#9B59B6',
    'pituitary': '#E67E22',
}

# Create directories
os.makedirs(CONFIG['FIGURE_PATH'], exist_ok=True)
os.makedirs(CONFIG['RESULTS_PATH'], exist_ok=True)

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['legend.fontsize'] = 11

# ============================================================================
# PAIRED DATA GENERATOR (for fusion)
# ============================================================================

class PairedMultimodalGenerator(tf.keras.utils.Sequence):
    """
    Generates batches of (MRI, CT) pairs with the same patient ID.
    Assumes filenames follow the pattern: <modality>_<label>_<patientID>_<tumorType?>_processed.npy
    For normal images, tumorType may be absent.
    """
    def __init__(self, mri_root, ct_root, split='train', batch_size=16,
                 mri_shape=(128,128,4), ct_shape=(224,224,4), shuffle=True):
        self.mri_root = os.path.join(mri_root, split)
        self.ct_root = os.path.join(ct_root, split)
        self.batch_size = batch_size
        self.mri_shape = mri_shape
        self.ct_shape = ct_shape
        self.shuffle = shuffle
        self.pairs = []          # list of (mri_path, ct_path, label, patient_id)
        self._build_pairs()
        self.on_epoch_end()

    def _extract_patient_id(self, filename):
        """Extract patient ID from filename (e.g., 'MRI_tumor_0034_2_processed.npy' → '0034')"""
        # Remove extension and split
        base = filename.replace('.npy', '')
        parts = base.split('_')
        # Pattern: modality_label_patientID_tumorType_processed (tumorType may be absent for normal)
        # Patient ID is the third part if it's a number
        if len(parts) >= 3 and parts[2].isdigit():
            return parts[2]
        # Fallback: try to find any numeric part
        numbers = re.findall(r'\d+', base)
        return numbers[0] if numbers else None

    def _build_pairs(self):
        """Scan both directories and match files by patient ID."""
        # Gather all MRI files
        mri_files = {}
        for root, _, files in os.walk(self.mri_root):
            for f in files:
                if f.endswith('.npy'):
                    patient_id = self._extract_patient_id(f)
                    if patient_id:
                        mri_files[patient_id] = os.path.join(root, f)

        # Gather all CT files and pair them
        for root, _, files in os.walk(self.ct_root):
            for f in files:
                if f.endswith('.npy'):
                    patient_id = self._extract_patient_id(f)
                    if patient_id and patient_id in mri_files:
                        mri_path = mri_files[patient_id]
                        ct_path = os.path.join(root, f)
                        # Determine label (0=normal, 1=tumor) from folder name or filename
                        if 'normal' in ct_path.lower():
                            label = 0
                        elif 'tumor' in ct_path.lower():
                            label = 1
                        else:
                            # Fallback: check filename
                            label = 1 if 'tumor' in f.lower() else 0
                        self.pairs.append((mri_path, ct_path, label, patient_id))

        if len(self.pairs) == 0:
            raise RuntimeError("No paired MRI-CT files found. Check your directory structure and patient ID extraction logic.")

        print(f"[PairedGenerator] Found {len(self.pairs)} pairs for split '{split}'")

    def __len__(self):
        return int(np.ceil(len(self.pairs) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = min((index + 1) * self.batch_size, len(self.pairs))
        batch_indices = self.indexes[start:end]
        batch_pairs = [self.pairs[i] for i in batch_indices]

        mri_batch = np.zeros((len(batch_pairs), *self.mri_shape), dtype=np.float32)
        ct_batch = np.zeros((len(batch_pairs), *self.ct_shape), dtype=np.float32)
        labels = []

        for i, (mri_path, ct_path, label, _) in enumerate(batch_pairs):
            mri_img = np.load(mri_path).astype(np.float32)
            ct_img = np.load(ct_path).astype(np.float32)

            # Ensure correct shapes (resize if needed)
            if mri_img.shape[:2] != self.mri_shape[:2]:
                mri_img = tf.image.resize(mri_img, self.mri_shape[:2]).numpy()
            if ct_img.shape[:2] != self.ct_shape[:2]:
                ct_img = tf.image.resize(ct_img, self.ct_shape[:2]).numpy()

            mri_batch[i] = mri_img
            ct_batch[i] = ct_img
            labels.append(label)

        labels = to_categorical(labels, num_classes=2)
        return (mri_batch, ct_batch), labels

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.pairs))
        if self.shuffle:
            np.random.shuffle(self.indexes)

# ============================================================================
# LOAD PRE-TRAINED MODELS AND CREATE FEATURE EXTRACTORS
# ============================================================================

def load_pretrained_models():
    """Load MRI stage-1, MRI stage-2, and CT models."""
    print("[1/6] Loading pre-trained models...")

    # MRI stage-1 (binary)
    mri_stage1 = tf.keras.models.load_model(CONFIG['MRI_STAGE1_PATH'])
    print("  - MRI stage-1 loaded.")

    # MRI stage-2 (multiclass)
    mri_stage2 = tf.keras.models.load_model(CONFIG['MRI_STAGE2_PATH'])
    print("  - MRI stage-2 loaded.")

    # CT model (needs custom CorrelationLayer)
    class CorrelationLayer(layers.Layer):
        def call(self, inputs):
            return tf.matmul(inputs, inputs, transpose_b=True)

    ct_model = tf.keras.models.load_model(
        CONFIG['CT_MODEL_PATH'],
        custom_objects={'CorrelationLayer': CorrelationLayer}
    )
    print("  - CT model loaded.")

    return mri_stage1, mri_stage2, ct_model

def create_feature_extractors(mri_stage1, ct_model):
    """
    Create models that output features from the penultimate layer.
    For MRI stage-1, we use the layer named 'fc3' (a Dense(256)).
    For CT, we use the penultimate Dense layer (32 units) – it has no name, so we index.
    """
    # MRI feature extractor: from input to 'fc3'
    mri_feature = models.Model(
        inputs=mri_stage1.input,
        outputs=mri_stage1.get_layer('fc3').output,
        name="MRI_feature_extractor"
    )
    # Freeze the weights (optional – you may fine-tune later)
    mri_feature.trainable = False
    print("  - MRI feature extractor created (output shape: 256).")

    # CT feature extractor: output of the last Dense layer before softmax (index -2)
    # We'll take the layer just before the final Dense(2). In your model summary,
    # that's the Dense(32) layer. If it's unnamed, we can get it by index.
    # Let's assume it's the second last layer.
    ct_feature = models.Model(
        inputs=ct_model.input,
        outputs=ct_model.layers[-2].output,   # penultimate layer (Dense(32))
        name="CT_feature_extractor"
    )
    ct_feature.trainable = False
    print("  - CT feature extractor created (output shape: 32).")

    return mri_feature, ct_feature

# ============================================================================
# BUILD FUSION MODEL
# ============================================================================

def build_fusion_model(mri_feature, ct_feature):
    """Concatenate features and add dense layers for binary classification."""
    mri_input = layers.Input(shape=CONFIG['FUSION_INPUT_SHAPE_MRI'], name="MRI_input")
    ct_input = layers.Input(shape=CONFIG['FUSION_INPUT_SHAPE_CT'], name="CT_input")

    # Extract features
    mri_feat = mri_feature(mri_input)   # shape: (256,)
    ct_feat = ct_feature(ct_input)       # shape: (32,)

    # Concatenate
    concat = layers.Concatenate(name="concat_features")([mri_feat, ct_feat])   # shape: (288,)

    # Joint representation layers
    x = layers.Dense(128, activation='relu', name="fc_joint_1")(concat)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu', name="fc_joint_2")(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(2, activation='softmax', name="output")(x)

    model = models.Model(inputs=[mri_input, ct_input], outputs=output, name="Multimodal_Fusion")
    return model

# ============================================================================
# TRAINING AND EVALUATION UTILITIES
# ============================================================================

def plot_fusion_training_curves(history):
    """Plot accuracy and loss for fusion model."""
    acc = np.array(history.history['accuracy']) * 100
    val_acc = np.array(history.history['val_accuracy']) * 100
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = np.arange(1, len(acc)+1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    fig.patch.set_facecolor(COLORS['background'])

    # Accuracy
    ax1.plot(epochs, acc, color=COLORS['multimodal'], label='Training Accuracy', lw=3)
    ax1.plot(epochs, val_acc, color='red', label='Validation Accuracy', lw=3)
    ax1.fill_between(epochs, acc, val_acc, alpha=0.1, color='gray')
    ax1.set_xlabel('Epochs', fontweight='bold')
    ax1.set_ylabel('Accuracy (%)', fontweight='bold')
    ax1.set_title('Fusion Model: Accuracy', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax1.set_ylim([min(acc.min(), val_acc.min())-5, 101])

    # Loss
    ax2.plot(epochs, loss, color=COLORS['multimodal'], label='Training Loss', lw=3)
    ax2.plot(epochs, val_loss, color='red', label='Validation Loss', lw=3)
    ax2.fill_between(epochs, loss, val_loss, alpha=0.1, color='gray')
    ax2.set_xlabel('Epochs', fontweight='bold')
    ax2.set_ylabel('Loss', fontweight='bold')
    ax2.set_title('Fusion Model: Loss', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    plt.suptitle('Multimodal Late Fusion Training History', fontsize=18, fontweight='bold')
    plt.savefig(os.path.join(CONFIG['FIGURE_PATH'], 'fusion_training_curves.png'), dpi=300)
    plt.show()

def plot_fusion_confusion_matrix(y_true, y_pred, class_names):
    """Confusion matrix for fusion model."""
    cm = confusion_matrix(y_true, y_pred)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.patch.set_facecolor(COLORS['background'])

    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=class_names, yticklabels=class_names, ax=ax1,
                cbar_kws={'label': 'Count', 'shrink': 0.8},
                annot_kws={'size': 14, 'weight': 'bold'})
    ax1.set_xlabel('Predicted', fontweight='bold')
    ax1.set_ylabel('True', fontweight='bold')
    ax1.set_title('Fusion: Raw Counts', fontweight='bold', pad=20)

    sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Purples',
                xticklabels=class_names, yticklabels=class_names, ax=ax2,
                cbar_kws={'label': 'Percentage (%)', 'shrink': 0.8},
                annot_kws={'size': 14, 'weight': 'bold'})
    ax2.set_xlabel('Predicted', fontweight='bold')
    ax2.set_ylabel('True', fontweight='bold')
    ax2.set_title('Fusion: Percentages', fontweight='bold', pad=20)

    # Add metrics
    tn, fp, fn, tp = cm.ravel()
    acc = (tp+tn)/np.sum(cm)
    sens = tp/(tp+fn) if (tp+fn)>0 else 0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0
    prec = tp/(tp+fp) if (tp+fp)>0 else 0
    f1 = 2*(prec*sens)/(prec+sens) if (prec+sens)>0 else 0

    textstr = (f'Accuracy:  {acc*100:.2f}%\n'
               f'Sensitivity: {sens*100:.2f}%\n'
               f'Specificity: {spec*100:.2f}%\n'
               f'Precision: {prec*100:.2f}%\n'
               f'F1-score:  {f1*100:.2f}%')
    ax2.text(1.3, 0.5, textstr, transform=ax2.transAxes,
             fontsize=12, fontweight='bold', verticalalignment='center',
             bbox=dict(boxstyle="round,pad=0.5", facecolor=COLORS['multimodal'], alpha=0.2))

    plt.suptitle('Multimodal Fusion: Confusion Matrix', fontsize=18, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['FIGURE_PATH'], 'fusion_confusion_matrix.png'), dpi=300)
    plt.show()
    return acc, sens, spec, prec, f1

def plot_fusion_roc_curve(y_true, y_pred_proba):
    """ROC curve with 95% CI via bootstrapping."""
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba[:, 1])
    base_auc = auc(fpr, tpr)

    # Bootstrapping for CI
    rng = np.random.RandomState(42)
    n_bootstraps = 1000
    bootstrapped_aucs = []
    y_true_arr = np.array(y_true)
    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_true_arr), len(y_true_arr))
        if len(np.unique(y_true_arr[indices])) < 2:
            continue
        score = roc_auc_score(y_true_arr[indices], y_pred_proba[indices, 1])
        bootstrapped_aucs.append(score)
    ci_lower = np.percentile(bootstrapped_aucs, 2.5)
    ci_upper = np.percentile(bootstrapped_aucs, 97.5)

    fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
    fig.patch.set_facecolor(COLORS['background'])

    ax.plot(fpr, tpr, color=COLORS['multimodal'], lw=4,
            label=f'Fusion (AUC = {base_auc:.3f})')
    ax.fill_between(fpr, tpr, alpha=0.2, color=COLORS['multimodal'],
                    label=f'95% CI: [{ci_lower:.3f}–{ci_upper:.3f}]')
    ax.plot([0,1], [0,1], 'k--', lw=2, label='Random')
    ax.set_xlabel('False Positive Rate', fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontweight='bold')
    ax.set_title('Multimodal Fusion: ROC Curve', fontweight='bold', pad=15)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

    # Inset
    axins = inset_axes(ax, width="30%", height="30%", loc='center right',
                       bbox_to_anchor=(0.05,0.05,0.9,0.9), bbox_transform=ax.transAxes)
    mask = fpr <= 0.2
    axins.plot(fpr[mask], tpr[mask], color=COLORS['multimodal'], lw=2)
    axins.plot([0,0.2], [0.95,0.95], 'k--', alpha=0.3)
    axins.set_xlim([0,0.2]); axins.set_ylim([0.8,1.0])
    axins.grid(True, alpha=0.3)
    mark_inset(ax, axins, loc1=1, loc2=3, fc="none", ec="gray")

    plt.savefig(os.path.join(CONFIG['FIGURE_PATH'], 'fusion_roc_curve.png'), dpi=300)
    plt.show()
    return base_auc, ci_lower, ci_upper

def plot_radar_chart(mri_scores, ct_scores, fusion_scores):
    """Radar chart comparing the three models."""
    categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']
    N = len(categories)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(10,10), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor(COLORS['background'])
    ax.set_facecolor(COLORS['background'])

    # Add metrics (ensure they are percentages)
    mri_vals = mri_scores + mri_scores[:1]
    ct_vals = ct_scores + ct_scores[:1]
    fusion_vals = fusion_scores + fusion_scores[:1]

    ax.plot(angles, mri_vals, linewidth=3, color=COLORS['mri'], label='MRI Only')
    ax.fill(angles, mri_vals, color=COLORS['mri'], alpha=0.1)
    ax.plot(angles, ct_vals, linewidth=3, color=COLORS['ct'], label='CT Only')
    ax.fill(angles, ct_vals, color=COLORS['ct'], alpha=0.1)
    ax.plot(angles, fusion_vals, linewidth=4, color=COLORS['multimodal'], label='Multimodal Fusion')
    ax.fill(angles, fusion_vals, color=COLORS['multimodal'], alpha=0.2)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=13, fontweight='bold')
    ax.set_ylim(80, 102)
    ax.set_yticks([80,85,90,95,100])
    ax.set_yticklabels(['80','85','90','95','100'], color='grey', size=10)
    ax.set_title('Performance Comparison: Unimodal vs. Multimodal', size=18, fontweight='bold', pad=30)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3,1.1), fontsize=12)

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['FIGURE_PATH'], 'radar_chart.png'), dpi=300)
    plt.savefig(os.path.join(CONFIG['FIGURE_PATH'], 'radar_chart.pdf'))
    plt.show()

# ============================================================================
# GRAD-CAM FOR INTERPRETABILITY
# ============================================================================

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Generate Grad-CAM heatmap."""
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    last_conv_output = last_conv_output[0]
    heatmap = last_conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, heatmap, alpha=0.4):
    """Overlay heatmap on original image (first channel)."""
    img = np.load(img_path)
    base_img = img[:,:,0]  # first channel (structural)
    base_img = np.uint8(255 * (base_img - base_img.min()) / (base_img.max() - base_img.min() + 1e-8))
    base_img = cv2.cvtColor(base_img, cv2.COLOR_GRAY2RGB)

    heatmap = np.uint8(255 * heatmap)
    jet = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((base_img.shape[1], base_img.shape[0]))
    jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap)

    superimposed = jet_heatmap * alpha + base_img
    superimposed = np.clip(superimposed, 0, 255).astype(np.uint8)
    return superimposed

def visualize_gradcam_for_sample(mri_path, ct_path, mri_model, ct_model, mri_layer, ct_layer):
    """Generate and display Grad-CAM for both modalities."""
    # Load and preprocess images
    mri_img = np.load(mri_path).astype(np.float32)
    ct_img = np.load(ct_path).astype(np.float32)
    if mri_img.shape[:2] != CONFIG['FUSION_INPUT_SHAPE_MRI'][:2]:
        mri_img = tf.image.resize(mri_img, CONFIG['FUSION_INPUT_SHAPE_MRI'][:2]).numpy()
    if ct_img.shape[:2] != CONFIG['FUSION_INPUT_SHAPE_CT'][:2]:
        ct_img = tf.image.resize(ct_img, CONFIG['FUSION_INPUT_SHAPE_CT'][:2]).numpy()
    mri_img_batch = np.expand_dims(mri_img, 0)
    ct_img_batch = np.expand_dims(ct_img, 0)

    # Predictions (for title)
    mri_pred = mri_model.predict(mri_img_batch, verbose=0)[0]
    ct_pred = ct_model.predict(ct_img_batch, verbose=0)[0]
    mri_conf = mri_pred[1]  # tumor probability
    ct_conf = ct_pred[1]

    # Generate heatmaps (for tumor class, index 1)
    mri_heatmap = make_gradcam_heatmap(mri_img_batch, mri_model, mri_layer, pred_index=1)
    ct_heatmap = make_gradcam_heatmap(ct_img_batch, ct_model, ct_layer, pred_index=1)

    # Overlay
    mri_cam = display_gradcam(mri_path, mri_heatmap)
    ct_cam = display_gradcam(ct_path, ct_heatmap)

    # Plot side by side
    fig, axes = plt.subplots(1, 2, figsize=(14,6))
    axes[0].imshow(mri_cam)
    axes[0].set_title(f"MRI Grad-CAM (Tumor conf: {mri_conf:.1%})", fontweight='bold')
    axes[0].axis('off')
    axes[1].imshow(ct_cam)
    axes[1].set_title(f"CT Grad-CAM (Tumor conf: {ct_conf:.1%})", fontweight='bold')
    axes[1].axis('off')
    plt.suptitle("Grad-CAM Localisation of Tumour Regions", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['FIGURE_PATH'], 'gradcam_example.png'), dpi=300)
    plt.show()

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def run_fusion_pipeline():
    print("="*70)
    print("🧠 MULTIMODAL LATE FUSION (FEATURE-LEVEL) PIPELINE")
    print("="*70)

    # 1. Load pre-trained models
    mri_stage1, mri_stage2, ct_model = load_pretrained_models()

    # 2. Create feature extractors
    mri_feature, ct_feature = create_feature_extractors(mri_stage1, ct_model)

    # 3. Build fusion model
    fusion_model = build_fusion_model(mri_feature, ct_feature)
    fusion_model.compile(
        optimizer=optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE']),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    fusion_model.summary()

    # 4. Prepare paired data generators
    print("[2/6] Preparing paired data generators...")
    train_gen = PairedMultimodalGenerator(
        CONFIG['MRI_DATA_ROOT'], CONFIG['CT_DATA_ROOT'], split='train',
        batch_size=CONFIG['BATCH_SIZE'],
        mri_shape=CONFIG['FUSION_INPUT_SHAPE_MRI'],
        ct_shape=CONFIG['FUSION_INPUT_SHAPE_CT']
    )
    val_gen = PairedMultimodalGenerator(
        CONFIG['MRI_DATA_ROOT'], CONFIG['CT_DATA_ROOT'], split='val',
        batch_size=CONFIG['BATCH_SIZE'],
        mri_shape=CONFIG['FUSION_INPUT_SHAPE_MRI'],
        ct_shape=CONFIG['FUSION_INPUT_SHAPE_CT'],
        shuffle=False
    )
    test_gen = PairedMultimodalGenerator(
        CONFIG['MRI_DATA_ROOT'], CONFIG['CT_DATA_ROOT'], split='test',
        batch_size=CONFIG['BATCH_SIZE'],
        mri_shape=CONFIG['FUSION_INPUT_SHAPE_MRI'],
        ct_shape=CONFIG['FUSION_INPUT_SHAPE_CT'],
        shuffle=False
    )

    # 5. Train fusion model
    print("[3/6] Training fusion model...")
    history = fusion_model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=CONFIG['EPOCHS'],
        verbose=1,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
        ]
    )
    fusion_model.save(CONFIG['FUSION_MODEL_SAVE_PATH'])
    plot_fusion_training_curves(history)

    # 6. Evaluate fusion model on test set
    print("[4/6] Evaluating fusion model on test set...")
    y_true, y_pred_proba = [], []
    for i in range(len(test_gen)):
        (mri_batch, ct_batch), y_batch = test_gen[i]
        preds = fusion_model.predict_on_batch([mri_batch, ct_batch])
        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred_proba.extend(preds)
    y_true = np.array(y_true)
    y_pred_proba = np.array(y_pred_proba)
    y_pred = np.argmax(y_pred_proba, axis=1)

    # Compute metrics
    acc, sens, spec, prec, f1 = plot_fusion_confusion_matrix(y_true, y_pred, CLASS_NAMES)
    roc_auc, ci_low, ci_high = plot_fusion_roc_curve(y_true, y_pred_proba)

    fusion_scores = [acc*100, prec*100, sens*100, f1*100, roc_auc*100]

    # 7. Load MRI and CT baseline scores (from previously saved pickle files)
    print("[5/6] Loading unimodal baseline scores...")
    with open('mri_results.pkl', 'rb') as f:
        mri_data = pickle.load(f)
    mri_scores = [mri_data['accuracy'], mri_data['precision'], mri_data['recall'],
                  mri_data['f1'], mri_data['auc']]

    # CT scores are not saved in the CT script? Let's compute them from test set using CT model alone.
    # For a fair comparison, we evaluate the CT model on the same paired test set (using only CT images)
    ct_test_preds = []
    ct_test_true = []
    for i in range(len(test_gen)):
        (_, ct_batch), y_batch = test_gen[i]
        preds = ct_model.predict_on_batch(ct_batch)
        ct_test_preds.extend(np.argmax(preds, axis=1))
        ct_test_true.extend(np.argmax(y_batch, axis=1))
    ct_test_true = np.array(ct_test_true)
    ct_test_preds = np.array(ct_test_preds)
    cm_ct = confusion_matrix(ct_test_true, ct_test_preds)
    tn, fp, fn, tp = cm_ct.ravel()
    ct_acc = (tp+tn)/np.sum(cm_ct)
    ct_sens = tp/(tp+fn) if (tp+fn)>0 else 0
    ct_prec = tp/(tp+fp) if (tp+fp)>0 else 0
    ct_f1 = 2*(ct_prec*ct_sens)/(ct_prec+ct_sens) if (ct_prec+ct_sens)>0 else 0
    # Need CT AUC: get probabilities
    ct_proba = []
    for i in range(len(test_gen)):
        (_, ct_batch), _ = test_gen[i]
        proba = ct_model.predict_on_batch(ct_batch)
        ct_proba.extend(proba)
    ct_proba = np.array(ct_proba)
    ct_auc = roc_auc_score(ct_test_true, ct_proba[:,1])
    ct_scores = [ct_acc*100, ct_prec*100, ct_sens*100, ct_f1*100, ct_auc*100]

    print("MRI scores:", mri_scores)
    print("CT scores:", ct_scores)
    print("Fusion scores:", fusion_scores)

    # 8. Radar chart
    plot_radar_chart(mri_scores, ct_scores, fusion_scores)

    # 9. Classification report
    print("\n" + "="*70)
    print("CLASSIFICATION REPORT - FUSION MODEL")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    # 10. Grad-CAM on a few test samples
    print("[6/6] Generating Grad-CAM examples...")
    # Find one tumor sample from test set
    tumor_idx = None
    for idx, (_, _, label, _) in enumerate(test_gen.pairs):
        if label == 1:
            tumor_idx = idx
            break
    if tumor_idx is not None:
        mri_path, ct_path, _, _ = test_gen.pairs[tumor_idx]
        # Determine last conv layer names (modify based on your actual models)
        mri_last_conv = 'conv2d_4'  # last Conv2D layer in MRI stage-1 (check model.summary)
        ct_last_conv = 'conv2d_7'    # last Conv2D in CT shared encoder (check model.summary)
        visualize_gradcam_for_sample(mri_path, ct_path, mri_stage1, ct_model,
                                     mri_last_conv, ct_last_conv)
    else:
        print("No tumor sample found in test set for Grad-CAM.")

    print("\n" + "="*70)
    print("✅ MULTIMODAL FUSION PIPELINE COMPLETED")
    print(f"All figures saved to: {CONFIG['FIGURE_PATH']}")
    print("="*70)

if __name__ == "__main__":
    run_fusion_pipeline()